# Lesson 8 — Exception Handling

Robust input handling: `try/except/else/finally`, specific exception
types, and raising to enforce contracts.

## How to work

- Write every solution in `solution.py` first, then paste the finished
  code into the matching cell and run it.
- **Rule for every exercise:** your printed output must match the
  `Output:` block **exactly** — error messages are part of the contract
  now.
- Read the `Constraint:` line. Violating it is a fail, even if the
  output looks right.
- Functions never print unless the task says so; printing happens in
  the provided driver/caller code.
- Hints are conceptual keywords, not answers — ask for one at a time.
- H1 ships broken code on purpose: find and fix the bugs; don't rewrite
  working parts.

Difficulty climbs after the Easy set.

## E1 — Guard the conversion

**Type:** [REAL-WORLD]

**Task:** write `to_int(text)` that returns the int value of `text`, or
the string `"invalid"` when conversion fails. Then run the driver:

```python
for t in ["42", "12a", ""]:
    print(to_int(t))
```

**Output:**
```
42
invalid
invalid
```

**Constraint:** use `try/except` around `int()`. No `isdigit()` /
`isnumeric()` / regex pre-checks. `to_int` must not print and must
never crash, whatever string it gets.

**Hints (ask one at a time):**
- Which single exception type does `int()` raise on bad text?
- The function has two exits, one per outcome. What does each return?
- Where does printing belong here — inside the function or in the loop?

In [4]:
def to_int(text):
    try:
        return int(text)
    except ValueError:
        return "invalid"
    
for t in ["42", "12a", ""]:
    print(to_int(t))

42
invalid
invalid


## E2 — Two different failures

**Type:** [REAL-WORLD]

**Task:** write `safe_divide(a, b)` returning the division result.
Handle the two failure modes separately: division by zero → return
`"undefined"`; wrong operand types → return `"type error"`. Driver:

```python
print(safe_divide(10, 4))
print(safe_divide(10, 0))
print(safe_divide(10, "x"))
```

**Output:**
```
2.5
undefined
type error
```

**Constraint:** two separate `except` blocks naming specific types.
No `except Exception`. No `if` checks replacing exception handling.

**Hints (ask one at a time):**
- Two different things can go wrong — name the exception each raises.
- Does clause order matter here? What could a broad first clause hide?
- One `try` may carry several `except` clauses — each answers one alarm.

In [5]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return "undefined"
    except TypeError:
        return "type error"
    
print(safe_divide(10,4))
print(safe_divide(10,0))
print(safe_divide(10,"x"))

2.5
undefined
type error


## E3 — else earns its keep

**Type:** [REAL-WORLD]

**Task:** write `load_port(config)` that reads `config["port"]` and
converts it to `int`. On success return `f"ready on {port}"`; missing
key → `"missing key"`; unparsable value → `"not a number"`. Driver:

```python
print(load_port({"port": "8080"}))
print(load_port({}))
print(load_port({"port": "abc"}))
```

**Output:**
```
ready on 8080
missing key
not a number
```

**Constraint:** the success path (`return f"ready on {port}"`) must sit
in an `else:` block, not inside `try`. Both excepts name specific
types.

**Hints (ask one at a time):**
- Three outcomes, three blocks: which block runs only when nothing failed?
- A missing key and a failed conversion raise different exceptions — which is which?
- If the success `return` stays inside `try`, what else would the excepts accidentally cover?

In [1]:
def load_port(config):
    try:
        port = int(config["port"])
    except KeyError:
        return "missing key"
    except ValueError:
        return "not a number"
    else:
        return f"ready on {port}"
    
print(load_port({"port": "8080"}))    
print(load_port({}))    
print(load_port({"port": "abc"}))    

ready on 8080
missing key
not a number


## E4 — finally counts

**Type:** [REAL-WORLD]

**Task:** simulate resource cleanup with an event log. Write
`process(name, events)` that appends `f"processed {name}"` on success;
when `name == "bad"` the operation fails and it appends
`f"failed {name}"`; either way it appends `f"closed {name}"` via
`finally`. Driver:

```python
events = []
process("data", events)
process("bad", events)
for e in events:
    print(e)
```

**Output:**
```
processed data
closed data
failed bad
closed bad
```

**Constraint:** the `closed` line is appended in exactly ONE place: a
`finally` block. No duplicated append in both `try` and `except`.

**Hints (ask one at a time):**
- `finally` doesn't care how the `try` ended — success or failure, it runs.
- For a successful call, what appends first — and does `finally` come before or after the response blocks?
- A naive version appends "closed" in two places. Yours should have one.

In [3]:
def process(name, events: list):
    try:
        if name == "bad":
            raise ValueError(f"failed {name}")
    except ValueError as err:
        events.append(str(err))
    else:
        events.append(f"processed {name}")
    finally:
        events.append(f"closed {name}")
        
events = []
process("data", events)
process("bad", events)

for e in events:
    print(e)

processed data
closed data
failed bad
closed bad


## E5 — Raise the alarm

**Type:** [REAL-WORLD]

**Task:** write `withdraw(balance, amount)` returning the new balance,
but raising `ValueError("insufficient funds")` when
`amount > balance`. The caller handles it. Driver:

```python
try:
    print(withdraw(100, 30))
except ValueError as err:
    print(err)

try:
    print(withdraw(100, 150))
except ValueError as err:
    print(err)
```

**Output:**
```
70
insufficient funds
```

**Constraint:** `withdraw` raises — it never prints and never returns
`None` on bad input. Exactly one `except ValueError as err` per driver
block, printing `err`.

**Hints (ask one at a time):**
- Who detects the violation, and who decides what the user sees? Two different jobs.
- The message string lives where the check happens — why there and not in the caller?
- Compare with returning `None`: what would the caller silently do with `None`?

In [4]:
def withdraw(balance, amount):
    if amount > balance:
        raise ValueError("insufficient funds")
    return balance - amount

try:
    print(withdraw(100,30))
except ValueError as err:
    print(err)
try:
    print(withdraw(100, 150))
except ValueError as err:
    print(err)

70
insufficient funds


## M1 — Parse the ledger

**Type:** [REAL-WORLD]

**Task:** write `summarize(records)` taking strings of the form
`"name,amount"` (amount parseable by `float`, may start with `+` or
`-`; name non-empty). It returns a dict with keys `"total"` (sum of
valid amounts), `"valid"` and `"invalid"` (counts). A record is invalid
if it has no comma, an empty name, or an unparsable amount. Driver:

```python
records = [
    "coffee,-3.50",
    "salary,+2500.00",
    "rent:bad",
    "",
    "-12",
    "snacks,+4.20",
]
s = summarize(records)
print(f"total: {s['total']:.2f}")
print(f"valid: {s['valid']}")
print(f"invalid: {s['invalid']}")
```

**Output:**
```
total: 2500.70
valid: 3
invalid: 3
```

**Constraint:** use `split(",")` + length check for structure and
`try/except ValueError` around `float()` — no `isdigit`, no regex.
`summarize` never prints.

**Hints (ask one at a time):**
- Two questions per record: is the shape right, is the value right? Different tools answer each.
- `float("+2500.00")` works — let the converter decide, don't pre-filter signs.
- Counters update once per record — watch where you initialize them.

In [10]:
def summarize(records: list[str]):
    s = {
        "total": 0,
        "valid": 0,
        "invalid": 0
    }
    for record in records:
        try:
            name, amount = record.split(",")
            
            if not name:
                raise ValueError("invalid name")
            
            amount = float(amount)
                        
        except ValueError:
            s["invalid"] += 1
            
        else:
            s["valid"] += 1
            s["total"] += amount

    return s

records = [
    "coffee,-3.50",
    "salary,+2500.00",
    "rent:bad",
    "",
    "-12",
    "snacks,+4.20",
]
s = summarize(records)
print(f"total: {s['total']:.2f}")
print(f"valid: {s['valid']}")
print(f"invalid: {s['invalid']}")


total: 2500.70
valid: 3
invalid: 3


## M2 — First usable reading

**Type:** [REAL-WORLD]

**Task:** sensors emit candidate readings; some are garbage. Write
`first_valid(candidates)` looping over the list and returning the first
value convertible to `int`. If none converts, raise
`ValueError("no valid input")`. Driver:

```python
try:
    print(first_valid(["x", "", "7", "3"]))
except ValueError as err:
    print(err)

try:
    print(first_valid(["a", "b"]))
except ValueError as err:
    print(err)
```

**Output:**
```
7
no valid input
```

**Constraint:** the `try/except` sits INSIDE the loop; invalid
candidates are skipped, not returned. The raise happens only after the
loop ends empty-handed. Function never prints.

**Hints (ask one at a time):**
- Skipping one bad item and giving up after all items are two different events — which happens inside the loop, which after?
- When the loop finds nothing, where does control go next?
- A `return` inside a loop ends it early — is that the exit you want for a good reading?

In [8]:
def first_valid(candidates: list):
    for c in candidates:
        try:
            return int(c)
        except ValueError:
            continue

    raise ValueError("no valid input")

try:
    print(first_valid(["x", "", "7", "3"]))
except ValueError as err:
    print(err)

try:
    print(first_valid(["a", "b"]))
except ValueError as err:
    print(err)

7
no valid input


## M3 — Validate at the border

**Type:** [REAL-WORLD]

**Task:** a settings module receives raw string values. Write
`load_settings(raw)` converting according to this exact spec:
`{"retries": int, "rate": float}`. On any conversion failure raise
`ValueError(f"bad setting: {key}")`. On success return the typed dict.
Driver:

```python
try:
    print(load_settings({"retries": "3", "rate": "0.5"}))
except ValueError as err:
    print(err)

try:
    print(load_settings({"retries": "three", "rate": "0.5"}))
except ValueError as err:
    print(err)
```

**Output:**
```
{'retries': 3, 'rate': 0.5}
bad setting: retries
```

**Constraint:** build the result following the SPEC's key order
(retries, then rate). One generic handler at the caller; the function
raises with the exact message — never prints, never returns a partial
dict.

**Hints (ask one at a time):**
- Loop over the spec, not over the raw dict — the spec defines what MUST exist.
- One `try` around both conversions is fine — but then how do you know WHICH key failed?
- The error message needs the key name — make sure your loop variable reaches the `raise`.

In [87]:
def load_settings(raw: dict):
    spec = {
        "retries": int,
        "rate": float, 
    }
    
    result = {
        "retries": 0,
        "rate": 0.0
    }
    
    for key, value_type in spec.items():
        try:
            result[key] = value_type(raw[key])
        except (ValueError, KeyError):
            raise ValueError(f"bad setting: {key}")
    return result

try:
    print(load_settings({"retries": "3", "rate": "0.5"}))
except ValueError as err:
    print(err)

try:
    print(load_settings({"retries": "three", "rate": "0.5"}))
except ValueError as err:
    print(err)

{'retries': 3, 'rate': 0.5}
bad setting: retries


## H1 — Debug: four bugs in the weather batch

**Type:** [REAL-WORLD] · debug drill

This module converts Fahrenheit readings to Celsius, skips bad rows,
and computes a mean. It was written carelessly. Run it, compare
expected vs observed, find and fix **four** bugs.

**Expected output (after fixes):**
```
[21.1, 23.9]
skipped: 1
22.5
no data
```

**Observed when run as-is:** wrong values, silent losses, and a crash —
read the traceback, it names the first crime scene.

**Rules:**
- Fix each bug with the MINIMAL change. No rewriting working parts.
- Do not modify the driver block.
- After fixing, explain each bug: expected vs observed behavior, root cause.

**The four bug families to hunt (from our recurring-mistakes ledger):**
bare/silent exception swallowing · inverted validation condition ·
missing degenerate-path guard · state that never updates.

**Hints (ask one at a time):**
- Run it. Which line crashes first, and what does the `if` right above it promise vs deliver?
- Two functions are supposed to report failures — does either actually record anything when a row fails?
- `except:` alone catches more than bad data. Which two exceptions should never be silenced, and why?
- For every `if`, ask: with this exact condition, WHEN does the body run? Is that the situation the message describes?

In [91]:
def to_celsius(text):
    return round((float(text) - 32) * 5 / 9, 1)


def read_batch(readings):
    good = []
    for r in readings:
        try:
            good.append(to_celsius(r))
        except:
            pass
            
    if len(good) >= 0:
        return None
    return good


def count_skipped(readings):
    skipped = 0
    for r in readings:
        try:
            to_celsius(r)
        except:
            pass
    return skipped


def mean(values):
    return sum(values) / len(values)


# --- driver (do not change) ---
temps = ["70", "oops", "75"]
print(read_batch(temps))
print(count_skipped(temps))
print(mean([21.1, 23.9]))
try:
    print(mean([]))
except ValueError as err:
    print(err)

None
0
22.5


ZeroDivisionError: division by zero

In [43]:
def to_celsius(text):
    return round((float(text) - 32) * 5 / 9, 1)


def read_batch(readings):
    good = []
    for r in readings:
        try:
            good.append(to_celsius(r))
        except (ValueError, TypeError):
            pass
            
    if not good: # easier check if good ever have a value
        return None
    return good


def count_skipped(readings):
    skipped = 0
    for r in readings:
        try:
            to_celsius(r)
        except (ValueError, TypeError):
            skipped += 1 # increment the skipped value
    return f"skipped: {skipped}" # change to f string


def mean(values):
    if not values: # zero division guard
        raise ValueError("no data")
    return sum(values) / len(values)


# --- driver (do not change) ---
temps = ["70", "oops", "75"]
print(read_batch(temps))
print(count_skipped(temps))
print(mean([21.1, 23.9]))
try:
    print(mean([]))
except ValueError as err:
    print(err)

[21.1, 23.9]
skipped: 1
22.5
no data


## H2 — Strict CSV field parser

**Type:** [INTERVIEW]

**Task:** write `parse_line(line, types)` splitting `line` on commas
and converting field `i` according to `types[i]` (`str`, `int`, or
`float`). Violations raise `ValueError`:
- empty line (or only whitespace) → `ValueError("empty row")`
- field `i` fails its declared type → `ValueError(f"bad field: {i}")`
- `str` fields accept any non-empty text; an empty field counts as
  `bad field: i`.

Driver:

```python
specs = (str, int, float)
try:
    print(parse_line("alice,30,1.75", specs))
except ValueError as err:
    print(err)

try:
    print(parse_line("bob,abc,1.70", specs))
except ValueError as err:
    print(err)

try:
    print(parse_line("   ", specs))
except ValueError as err:
    print(err)
```

**Output:**
```
('alice', 30, 1.75)
bad field: 1
empty row
```

**Constraint:** no regex. Convert with the type object itself
(`types[i](value)`), wrapped in `try/except ValueError`. A `TypeError`
(wrong field count vs spec) should propagate — that's a programmer
error, not bad data. Return a tuple; never print.

**Hints (ask one at a time):**
- `int` and `float` ARE type objects — calling `types[i](raw)` converts without an if/elif ladder.
- Decide first: what makes a row "empty" before you even split it?
- The error message needs the field index — do you know it before or during the loop?

In [ ]:
def parse_line(line: str, types: tuple):
    line = line.strip()
    
    if not line: 
        raise ValueError("empty row")
    
    lines = line.split(",")
    result = []
    
    for i in range(len(types)):
        if types[i] == str and not lines[i]:
            raise ValueError(f"bad field: {i}")
        
        try:
            result.append(types[i](lines[i]))
        
        except ValueError:
            raise ValueError(f"bad field: {i}")
        
    return tuple(result)
        
specs = (str, int, float)
try:
    print(parse_line("alice,30,1.75", specs))
except ValueError as err:
    print(err)

try:
    print(parse_line("bob,abc,1.70", specs))
except ValueError as err:
    print(err)

try:
    print(parse_line("   ", specs))
except ValueError as err:
    print(err)

('alice', 30, 1.75)
bad field: 1
empty row


## Challenge — Expense Entry Validator CLI

**Type:** [REAL-WORLD] · micro build

First stone of the Stage 2 project. Build a small pipeline that parses
expense entries, rejects malformed ones, and prints a summary. Minimum
three functions:

- `parse_entry(line)` — expects `"description,amount"`. Violations raise
  `ValueError`: no comma → `"missing comma"`; description empty after
  `.strip()` → `"empty description"`; amount not float-parseable →
  `"bad amount"`. Success returns `(description, amount)` with the
  description stripped.
- `format_entry(description, amount)` — returns
  `f"{description} {amount:.2f}"`.
- `main(lines)` — loops, collects valid entries via `format_entry`,
  counts rejected ones, prints the report. ALL exception handling
  happens here, in one place.

Input lines:

```python
lines = [
    "coffee,-3.50",
    "salary,+2500.00",
    "nope",
    ",5.00",
    "snack,abc",
]
main(lines)
```

**Output:**
```
=== valid ===
coffee -3.50
salary 2500.00
=== rejected: 3 ===
total: 2496.50
```

**Constraints:**
- `parse_entry` and `format_entry` never print; `main` never converts floats itself.
- Exactly one `try/except ValueError` site, inside `main`.
- Report lines exactly as shown; `total` is the sum of valid amounts, formatted `.2f`.
- Rejection reasons are counted, not printed.

**Acceptance criteria:** runs clean on the given input AND on `[]`
(zero valid, `rejected: 0`, `total: 0.00`); every function does one
job; no duplicated parsing logic between functions.

**Stretch goals (optional):**
- Sort valid entries by amount ascending before printing.
- Track WHICH reason rejected each line; add a second report section.
- Make `main(lines, out=print)` injectable so tests can capture output.

**Hints (ask one at a time):**
- Design the data flow first: while looping, what does `main` hold — entries, rejections, running total?
- One rejection can happen for three reasons — where do those reasons live, and does `main` need to know which?
- Empty input is a degenerate path: decide what the report looks like before writing the loop.

In [ ]:
def parse_entry(line: str):
    if "," not in line:
        raise ValueError("missing comma")

    description, amount = (part.strip() for part in line.split(","))
    
    if not description:
        raise ValueError("empty description")
    
    try:
        amount = float(amount)
    except ValueError:
        raise ValueError("bad amount")

    return description, amount

def format_entry(description, amount):
    return f"{description} {amount:.2f}"

def main(lines, out=print):
    summary = {
        "rejected": 0,
        "total": 0,
        "valid": []
    }
    
    for line in lines:
        try:
            description, amount = parse_entry(line)
        except ValueError as err:
            summary["rejected"] += 1
        else:
            summary["total"] += amount
            summary["valid"].append(format_entry(description, amount))
        
    out("=== valid ===")
    out("\n".join(valid for valid in summary['valid']))
    out(f"=== rejected: {summary['rejected']} ===")
    out(f"total: {summary['total']:.2f}")
    
    return

lines = [
    "coffee,-3.50",
    "salary,+2500.00",
    "nope",
    ",5.00",
    "snack,abc",
]
main(lines)

=== valid ===
coffee -3.50
salary 2500.00
=== rejected: 3 === 
total: 2496.50
